# Full builds on Colab

Runs **everything** on Colab's free GPU, inside this notebook, with no tunnel: discovery, validation, quiz generation and the benchmark nights. It uses `qwen2.5:7b-instruct` and `llama3.1:8b`.
Step-by-step clicks and the files to upload or download: `docs/colab_steps.md` in the repo.

Run the cells in order. After a disconnect, run them again from the top: finished steps are skipped and saved LLM calls are read back from Drive.

In [ ]:
# Step 0: settings. Change STAGE, COURSE and STEPS only.
# course: one course | night1: benchmark discovery | night2, night3: benchmark validation
# real: ds101/ml101 with necessity at weight 1 (run after the course stage)
STAGE = "course"
COURSE = "ds101"  # ds101 and ml101 run necessity on 40 edges; any other course skips necessity
# Any of extract,discover,validate,quiz,path. Stop after discover to review before validation.
STEPS = "extract,discover"
COMMIT = "0b8a328c71bc74605aba3556dce07d103bbf6071"  # pinned repo commit
DISCOVERY_MODEL = "qwen2.5:7b-instruct"
VALIDATOR_MODEL = "llama3.1:8b"
DRIVE_DIR = "/content/drive/MyDrive/cdlp"
REPO_URL = "https://github.com/shreeya-g10/concept-dependency-learning-path.git"
REPO = "/content/cdlp"
assert STAGE in ("course", "night1", "night2", "night3", "real"), STAGE
assert len(COMMIT) == 40, "COMMIT must be a full 40-character commit SHA"

In [ ]:
# Step 1: is a GPU present? (Runtime > Change runtime type > T4 GPU)
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU > Save, then rerun.")
GPU = gpu.stdout.strip()
print("GPU:", GPU)

In [ ]:
# Step 2: install Ollama, start it, pull both models (rerun this cell after a runtime restart)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import time

import requests

server = subprocess.Popen(
    ["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT
)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("Ollama did not start: see /content/ollama.log")
!ollama pull {DISCOVERY_MODEL}
!ollama pull {VALIDATOR_MODEL}
!ollama --version
!ollama list

In [ ]:
# Step 3: clone the repo at the pinned commit, install what the modules need, and download the
# embedding model. Colab's disk is wiped between sessions, so this runs in every session.
import os

if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
!git -C {REPO} fetch -q origin
!git -C {REPO} checkout -q {COMMIT}
!git -C {REPO} log --oneline -1
!pip install -q -r {REPO}/cdlp/shared/requirements.txt -r {REPO}/cdlp/extraction/requirements.txt
!pip install -q -r {REPO}/cdlp/discovery/requirements.txt -r {REPO}/cdlp/validation/requirements.txt
!pip install -q -r {REPO}/cdlp/learning_path/requirements.txt
from sentence_transformers import SentenceTransformer

# On the CPU, so this cell holds no GPU memory while Ollama and the stages run.
dimension = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device="cpu"
).get_sentence_embedding_dimension()
print("embedding model ready, dimension:", dimension)

In [ ]:
# Step 4: Drive keeps outputs/ and the LLM cache, so a disconnect does not repeat saved calls
import glob
import shutil
import zipfile

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/cache/llm", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)
for zip_path in sorted(
    glob.glob(f"{DRIVE_DIR}/*_start.zip") + glob.glob(f"{DRIVE_DIR}/outputs.zip")
):
    # add newly uploaded files; never overwrite a file already on Drive
    with zipfile.ZipFile(zip_path) as z:
        new = [m for m in z.namelist() if not os.path.exists(f"{DRIVE_DIR}/{m}")]
        z.extractall(DRIVE_DIR, members=new)
    os.replace(zip_path, f"{zip_path}.added")
    print(f"added {len(new)} files from {os.path.basename(zip_path)}")
for link, target in (
    (f"{REPO}/outputs", f"{DRIVE_DIR}/outputs"),
    (f"{REPO}/.cache/llm", f"{DRIVE_DIR}/cache/llm"),
):
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.exists(link):
        shutil.rmtree(link)
    os.symlink(target, link)
os.environ.update(
    LLM_PROVIDER="ollama",
    LLM_BASE_URL="http://localhost:11434",
    LLM_MODEL=DISCOVERY_MODEL,
    LLM_VALIDATOR_MODEL=VALIDATOR_MODEL,
)
# A course that is not in the repo reads its files from My Drive/cdlp/raw/<course>/.
RAW = f"{DRIVE_DIR}/raw/{COURSE}"
RAW_ARG = f"--raw-dir {RAW}" if not os.path.isdir(f"{REPO}/data/raw/{COURSE}") else ""
print("outputs on Drive:", sorted(os.listdir(f"{DRIVE_DIR}/outputs")))
print("cached LLM calls on Drive:", len(os.listdir(f"{DRIVE_DIR}/cache/llm")))

In [ ]:
# Step 5: seconds per call: 10 short and 5 long for each model, in a throwaway cache
%cd {REPO}
!python scripts/colab_run.py stage speed

In [ ]:
# Step 6: run the stage. It stops if there is no GPU, records the GPU and both digests, and prints
# seconds per call per call type and the total time. After a disconnect: run steps 0-4, then this
# cell again. Finished steps are skipped; saved calls come back from Drive.
%cd {REPO}
if STAGE == "course":
    !python scripts/colab_run.py stage course --course {COURSE} --steps {STEPS} {RAW_ARG}
else:
    !python scripts/colab_run.py stage {STAGE}

In [ ]:
# Step 7: progress (any time), and for a course what it produced: decisions, drop rate per level
%cd {REPO}
if STAGE == "course":
    !python scripts/colab_run.py progress --stage course --course {COURSE} --steps {STEPS} {RAW_ARG}
    !python scripts/colab_run.py report --course {COURSE}
else:
    !python scripts/colab_run.py progress --stage {STAGE}

In [ ]:
# Step 8: zip for download. A course zip holds only that course's outputs, and this cell prints
# exactly where to put it so the website lists the course.
import datetime

%cd {REPO}
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
if STAGE == "course":
    !python scripts/colab_run.py zip --course {COURSE} --dest {DRIVE_DIR}
else:
    results = shutil.make_archive(
        f"{DRIVE_DIR}/results_{STAGE}_{stamp}", "zip", root_dir=DRIVE_DIR, base_dir="outputs"
    )
    print("Download from Drive:", results)
cache = shutil.make_archive(
    f"{DRIVE_DIR}/cache_{stamp}", "zip", root_dir=f"{DRIVE_DIR}/cache", base_dir="llm"
)
print("Saved LLM replies, optional, for local scoring:", cache)